In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report

In [4]:
pd.set_option("display.max_columns", None)

In [5]:
df = pd.read_csv('../raw_data/investments_VC.csv', encoding='latin1', low_memory=False)
print(df.shape)

(54294, 39)


In [6]:
# drops all rows with more than 50% missing data
row_missing_pct = df.isna().mean(axis=1).mul(100)
df = df[row_missing_pct <= 50]

print(f"New shape: {df.shape}")

New shape: (49438, 39)


In [7]:
# printing missing values
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct    dtype
state_code                    19277        38.99   object
founded_year                  10956        22.16  float64
founded_quarter               10956        22.16   object
founded_month                 10956        22.16   object
founded_at                    10884        22.02   object
city                           6116        12.37   object
country_code                   5273        10.67   object
region                         5273        10.67   object
 market                        3968         8.03   object
category_list                  3961         8.01   object
homepage_url                   3449         6.98   object
status                         1314         2.66   object
round_C                           0         0.00  float64
post_ipo_debt                     0         0.00  float64
secondary_market                  0         0.00  float64
product_crowdfunding              0         0.00  float64
round_A       

In [8]:
cols_to_drop = [
'city',
'founded_quarter', # can be derived from founded_at
'founded_month', # can be derived from founded_at
'founded_year', # can be derived from founded_at
'homepage_url',
'name',
]

df = df.drop(columns=cols_to_drop)
print(f"New shape: {df.shape}")

New shape: (49438, 33)


In [9]:
# Check current types
print(df.dtypes)

permalink                object
category_list            object
 market                  object
 funding_total_usd       object
status                   object
country_code             object
state_code               object
region                   object
funding_rounds          float64
founded_at               object
first_funding_at         object
last_funding_at          object
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A                 float64
round_B                 float64
round_C                 float64
round_D                 float64
round_E                 float64
round_F                 float64
round_G 

In [10]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].str.strip()   # remove whitespace
    df[col] = df[col].str.lower()   # standardise case

print("String columns cleaned")

String columns cleaned


In [11]:
# This fixes the column NAMES/HEADERS
df.columns = df.columns.str.strip()

In [12]:
print(df.dtypes)

permalink                object
category_list            object
market                   object
funding_total_usd        object
status                   object
country_code             object
state_code               object
region                   object
funding_rounds          float64
founded_at               object
first_funding_at         object
last_funding_at          object
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A                 float64
round_B                 float64
round_C                 float64
round_D                 float64
round_E                 float64
round_F                 float64
round_G 

In [13]:
# Convert all date columns from object to datetime
date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Clean funding_total_usd before converting to numeric
df['funding_total_usd'] = (df['funding_total_usd']
.str.replace(',', '', regex=False) # remove US style commas
.str.replace('-', '0', regex=False) # convert dashes to 0
.str.replace('$', '', regex=False) # remove currency symbols
)

df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')


# Verify the changes
print("\nAfter conversion:")
print(df[date_cols + ['funding_total_usd']].dtypes)
print(f"NaNs after conversion: {df['funding_total_usd'].isna().sum()}")
print(df['funding_total_usd'].describe())


After conversion:
founded_at           datetime64[ns]
first_funding_at     datetime64[ns]
last_funding_at      datetime64[ns]
funding_total_usd             int64
dtype: object
NaNs after conversion: 0
count    4.943800e+04
mean     1.316667e+07
std      1.535540e+08
min      0.000000e+00
25%      5.000000e+04
50%      1.000000e+06
75%      6.772162e+06
max      3.007950e+10
Name: funding_total_usd, dtype: float64


In [14]:
print(df.shape)

(49438, 33)


In [15]:
# How many companies founded before 2000
print(f"Companies founded before 2000: {(df['founded_at'] < '2000-01-01').sum()}")

Companies founded before 2000: 3730


In [16]:
df = df[(df['founded_at'] >= '2000-01-01') | (df['founded_at'].isna())]
print(f"New shape: {df.shape}")

New shape: (45708, 33)


In [17]:
# Checking for missing values again after updates to df were made
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct           dtype
state_code                    18353        40.15          object
founded_at                    10885        23.81  datetime64[ns]
country_code                   5128        11.22          object
region                         5128        11.22          object
market                         3645         7.97          object
category_list                  3638         7.96          object
status                         1173         2.57          object
first_funding_at                  9         0.02  datetime64[ns]
last_funding_at                   6         0.01  datetime64[ns]
round_C                           0         0.00         float64
secondary_market                  0         0.00         float64
product_crowdfunding              0         0.00         float64
round_A                           0         0.00         float64
round_B                           0         0.00         float64
permalink                

In [18]:
df.head(10)

,permalink,category_list,market,funding_total_usd,status,country_code,state_code,region,funding_rounds,founded_at,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,|entertainment|politics|social media|news|,news,1750000,acquired,usa,ny,new york city,1.0,2012-06-01,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,|games|,games,4000000,operating,usa,ca,los angeles,2.0,NaT,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,|publishing|education|,publishing,40000,operating,est,NaN,tallinn,1.0,2012-10-26,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/organization/in-touch-network,|electronics|guides|coffee|restaurants|music|i...,electronics,1500000,operating,gbr,NaN,london,1.0,2011-04-01,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/organization/r-ranch-and-mine,|tourism|entertainment|games|,tourism,60000,operating,usa,tx,dallas,2.0,2014-01-01,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,/organization/club-domains,|software|,software,7000000,NaN,usa,fl,ft. lauderdale,1.0,2011-10-10,2013-05-31,2013-05-31,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0
6,/organization/fox-networks,|advertising|,advertising,4912393,closed,arg,NaN,buenos aires,1.0,NaT,2007-01-16,2007-01-16,0.0,0.0,0.0,4912393.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,/organization/0-6-com,|curated web|,curated web,2000000,operating,NaN,NaN,NaN,1.0,2007-01-01,2008-03-19,2008-03-19,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,/organization/004-technologies,|software|,software,0,operating,usa,il,"springfield, illinois",1.0,2010-01-01,2014-07-24,2014-07-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,/organization/01games-technology,|games|,games,41250,operating,hkg,NaN,hong kong,1.0,NaT,2014-07-01,2014-07-01,41250.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# Categorical columns - fill with 'unknown'
cat_cols = ['state_code', 'country_code', 'region', 'market', 'category_list', 'status']
df[cat_cols] = df[cat_cols].fillna('unknown')


# Dates - fill with median
df['founded_at'] = df['founded_at'].fillna(df['founded_at'].median())
df['first_funding_at'] = df['first_funding_at'].fillna(df['first_funding_at'].median())
df['last_funding_at'] = df['last_funding_at'].fillna(df['last_funding_at'].median())

# Verify nothing is missing
print(df.isna().sum().sort_values(ascending=False))

permalink               0
debt_financing          0
round_G                 0
round_F                 0
round_E                 0
round_D                 0
round_C                 0
round_B                 0
round_A                 0
product_crowdfunding    0
secondary_market        0
post_ipo_debt           0
post_ipo_equity         0
private_equity          0
grant                   0
angel                   0
convertible_note        0
category_list           0
undisclosed             0
equity_crowdfunding     0
venture                 0
seed                    0
last_funding_at         0
first_funding_at        0
founded_at              0
funding_rounds          0
region                  0
state_code              0
country_code            0
status                  0
funding_total_usd       0
market                  0
round_H                 0
dtype: int64


In [20]:
df["status"].unique()

array(['acquired', 'operating', 'unknown', 'closed'], dtype=object)

In [21]:
df_preliminary_model = df.dropna(subset=["status"]).copy()

In [22]:
df_preliminary_model = df_preliminary_model.dropna(subset=["status"])

In [23]:
df["status"].value_counts(normalize=True)

status
operating    0.851973
acquired     0.067647
closed       0.054717
unknown      0.025663
Name: proportion, dtype: float64

In [24]:
X = df_preliminary_model.select_dtypes(include=["number"])
y = df_preliminary_model["status"].map({
    "acquired": 1,
    "operating": 1,
    "closed": 0
})

In [25]:
X.isnull().sum().sort_values(ascending=False).head()

funding_total_usd    0
post_ipo_debt        0
round_G              0
round_F              0
round_E              0
dtype: int64

In [26]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
pipeline.fit(X_train, y_train)

ValueError: Input y contains NaN.

In [29]:
pipeline.score(X_test, y_test)

NotFittedError: Pipeline is not fitted yet.

In [62]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.06      0.72      0.12       465
           1       0.96      0.41      0.58      8442

    accuracy                           0.43      8907
   macro avg       0.51      0.56      0.35      8907
weighted avg       0.92      0.43      0.55      8907



In [34]:
df_fe = df.copy()

In [35]:
date_cols = ["founded_at", "first_funding_at", "last_funding_at"]

for col in date_cols:
    df_fe[col] = pd.to_datetime(df_fe[col], errors="coerce")

In [36]:
df_fe["avg_raised_per_round"] = np.where(
    df_fe["funding_rounds"] > 0,
    df_fe["funding_total_usd"] / df_fe["funding_rounds"],
    np.nan
)

In [37]:
df_fe["age_first_funding_days"] = (
    df_fe["first_funding_at"] - df_fe["founded_at"]
).dt.days

In [38]:
df_fe["age_first_funding_days"] = df_fe["age_first_funding_days"].clip(lower=0)

In [39]:
df_fe["has_multiple_rounds"] = (df_fe["funding_rounds"] > 1).astype(int)

In [40]:
df_fe["funding_span_days"] = (
    df_fe["last_funding_at"] - df_fe["first_funding_at"]
).dt.days

In [41]:
df_fe["funding_span_days"] = df_fe["funding_span_days"].clip(lower=0)

In [42]:
df_fe["avg_years_between_rounds"] = np.where(
    df_fe["funding_rounds"] > 1,
    (df_fe["funding_span_days"] / 365.25) / (df_fe["funding_rounds"] - 1),
    np.nan
)

In [43]:
median_gap = df_fe.loc[
    df_fe["funding_rounds"] > 1, "avg_years_between_rounds"
].median()

df_fe["avg_years_between_rounds"] = df_fe["avg_years_between_rounds"].fillna(median_gap)

In [ ]:
eu_uk = {
    "gbr","deu","fra","esp","ita","nld","bel","swe","dnk","fin","irl",
    "aut","prt","pol","cze","hun","grc","rou","bgr","hrv","svk","svn",
    "est","lva","ltu","lux","mlt","cyp", "rom"
}

asia = {
    "chn","ind","jpn","kor","sgp","hkg","idn","mys","tha","vnm",
    "phl","twn","pak","bgd","lka","npl","isr","are","sau","kwt",
    "qat","omn","bhr"
}

americas = {
    "mex","bra","arg","chl","col","per","ury","ecu","bol","pry",
    "cri","pan","gtm","dom","jam","tto"
}

def map_region(country_code):
    if pd.isna(country_code):
        return "Unknown"

    if country_code == "USA":
        return "USA"
    if country_code == "CAN":
        return "Canada"
    if country_code in eu_uk:
        return "EU_UK"
    if country_code in asia:
        return "Asia"
    if country_code == "AUS":
        return "Australia"
    if country_code in americas:
        return "Rest_Americas"

    return "Rest_World"

df_fe["region_group"] = df_fe["country_code"].apply(map_region)

In [52]:
df_fe["country_code"].value_counts().head(20)

country_code
usa        26116
unknown     5128
gbr         2465
can         1275
chn         1145
deu          911
fra          798
ind          784
isr          634
esp          535
rus          357
aus          300
ita          299
nld          298
swe          296
sgp          292
irl          286
chl          285
bra          268
jpn          267
Name: count, dtype: int64

In [53]:
df_fe["market"].nunique()


751

In [56]:
df_fe["market_clean"].value_counts(normalize=True).head(20)

market_clean
software               0.088431
unknown                0.079745
biotechnology          0.071563
mobile                 0.041021
e-commerce             0.037368
curated web            0.034961
enterprise software    0.025072
games                  0.024591
clean technology       0.024153
health care            0.023913
advertising            0.022031
hardware + software    0.020959
social media           0.018903
health and wellness    0.018027
finance                0.017721
education              0.017109
manufacturing          0.013936
analytics              0.012842
security               0.010305
semiconductors         0.009408
Name: proportion, dtype: float64

In [55]:
df_fe["market_clean"] = (
    df_fe["market"]
    .str.strip()
    .str.lower()
)

In [57]:
len(df_fe["market_clean"].unique())

751

In [58]:
top_markets = df_fe["market_clean"].value_counts().head(30)
top_markets

market_clean
software               4042
unknown                3645
biotechnology          3271
mobile                 1875
e-commerce             1708
curated web            1598
enterprise software    1146
games                  1124
clean technology       1104
health care            1093
advertising            1007
hardware + software     958
social media            864
health and wellness     824
finance                 810
education               782
manufacturing           637
analytics               587
security                471
semiconductors          430
hospitality             425
consulting              400
fashion                 376
real estate             370
web hosting             365
news                    361
travel                  339
music                   282
messaging               281
search                  277
Name: count, dtype: int64

In [59]:
df_fe["market_clean"].value_counts(normalize=True).head(30)

market_clean
software               0.088431
unknown                0.079745
biotechnology          0.071563
mobile                 0.041021
e-commerce             0.037368
curated web            0.034961
enterprise software    0.025072
games                  0.024591
clean technology       0.024153
health care            0.023913
advertising            0.022031
hardware + software    0.020959
social media           0.018903
health and wellness    0.018027
finance                0.017721
education              0.017109
manufacturing          0.013936
analytics              0.012842
security               0.010305
semiconductors         0.009408
hospitality            0.009298
consulting             0.008751
fashion                0.008226
real estate            0.008095
web hosting            0.007985
news                   0.007898
travel                 0.007417
music                  0.006170
messaging              0.006148
search                 0.006060
Name: proportion, dtype: fl

In [61]:
top20_share = df_fe["market_clean"].value_counts(normalize=True).head(30).sum()
top20_share

np.float64(0.6881071147282751)

In [77]:
def map_industry(market):
    if pd.isna(market) or market == "unknown":
        return "Unknown"

    market = market.lower()

    if market in [
        "software", "enterprise software", "analytics",
        "web hosting", "security", "saas", "cloud computing",
        "big data", "internet", "technology"
    ]:
        return "Software_Data"

    if market in [
        "curated web", "social media", "messaging",
        "news", "music", "games", "apps", "video",
        "entertainment", "social network media", "photography",
        "search"
    ]:
        return "Consumer_Internet"

    if market in [
        "biotechnology", "health care", "health and wellness",
        "medical"
    ]:
        return "Health_Bio"

    if market in [
        "e-commerce", "marketplaces"
    ]:
        return "Ecommerce"

    if market in [
        "consulting", "advertising", "public relations",
        "sales and marketing", "design"
    ]:
        return "Services"

    if market in [
        "manufacturing", "real estate", "hospitality",
        "travel", "fashion", "automotive", "transportation",
        "sports"
    ]:
        return "Real_World"

    if market in [
        "hardware + software", "semiconductors", "networking"
    ]:
        return "Hardware_DeepTech"

    if market == "clean technology":
        return "Energy"

    if market == "finance":
        return "FinTech"

    if market == "education":
        return "Education"

    return "Other"

In [78]:
df_fe["industry_group"] = df_fe["market_clean"].apply(map_industry)

In [79]:
df_fe["industry_group"].value_counts(normalize=True)

industry_group
Other                0.272337
Software_Data        0.166623
Consumer_Internet    0.128490
Health_Bio           0.118798
Unknown              0.079745
Real_World           0.059268
Ecommerce            0.041765
Services             0.039643
Hardware_DeepTech    0.034348
Energy               0.024153
FinTech              0.017721
Education            0.017109
Name: proportion, dtype: float64

In [80]:
df_fe["industry_group"].value_counts(normalize=True)

industry_group
Other                0.272337
Software_Data        0.166623
Consumer_Internet    0.128490
Health_Bio           0.118798
Unknown              0.079745
Real_World           0.059268
Ecommerce            0.041765
Services             0.039643
Hardware_DeepTech    0.034348
Energy               0.024153
FinTech              0.017721
Education            0.017109
Name: proportion, dtype: float64

In [81]:
df_fe["industry_group"].unique()

array(['Consumer_Internet', 'Other', 'Software_Data', 'Services',
       'Ecommerce', 'Real_World', 'Education', 'Health_Bio', 'Unknown',
       'FinTech', 'Hardware_DeepTech', 'Energy'], dtype=object)

In [83]:
df_fe.head(5)

,permalink,category_list,market,funding_total_usd,status,country_code,state_code,region,funding_rounds,founded_at,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H,avg_raised_per_round,age_first_funding_days,has_multiple_rounds,funding_span_days,avg_years_between_rounds,region_group,market_clean,industry_group
0,/organization/waywire,|entertainment|politics|social media|news|,news,1750000,acquired,usa,ny,new york city,1.0,2012-06-01,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1750000.0,29,0,0,1.055578,Rest_World,news,Consumer_Internet
1,/organization/tv-communications,|games|,games,4000000,operating,usa,ca,los angeles,2.0,2010-02-01,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000000.0,123,1,111,0.303901,Rest_World,games,Consumer_Internet
2,/organization/rock-your-paper,|publishing|education|,publishing,40000,operating,est,unknown,tallinn,1.0,2012-10-26,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40000.0,0,0,0,1.055578,EU_UK,publishing,Other
3,/organization/in-touch-network,|electronics|guides|coffee|restaurants|music|i...,electronics,1500000,operating,gbr,unknown,london,1.0,2011-04-01,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1500000.0,0,0,0,1.055578,EU_UK,electronics,Other
4,/organization/r-ranch-and-mine,|tourism|entertainment|games|,tourism,60000,operating,usa,tx,dallas,2.0,2014-01-01,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30000.0,228,1,40,0.109514,Rest_World,tourism,Other
